In [1]:
!pip install openai-whisper sounddevice scipy

In [2]:
!pip install openai

In [3]:
!pip install pandas scikit-learn joblib

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Definiera den absoluta sökvägen till din data-mapp
DATA_PATH = r"C:\Users\tommi\Desktop\project_e-da\data\intent_dataset.csv"

# 1. Ladda in datasetet från din specifika mapp
df = pd.read_csv(DATA_PATH)

# Fortsätt sedan med uppdelningen precis som förut
X = df['text']
y = df['label']

# 3. Dela upp datan i tränings- och testset (80% träning, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Text-förbehandling (Vektorisering)
# Datorn förstår inte text, så vi översätter orden till siffror (TF-IDF)
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 5. Träna modellen (Vi använder Logistic Regression för textklassificering)
model = LogisticRegression()
model.fit(X_train_vec, y_train)

# 6. Utvärdera modellen (För att nå VG-kriteriet!)
y_pred = model.predict(X_test_vec)
print(f"Modellens träffsäkerhet (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nDetaljerad utvärdering:")
print(classification_report(y_test, y_pred))

# 7. Spara modellen och vektoriseraren så vi kan använda dem i din app.py senare
joblib.dump(model, 'intent_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("\n✅ Modellen och vektoriseraren är sparade!")

Modellens träffsäkerhet (Accuracy): 75.00%

Detaljerad utvärdering:
              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.67      1.00      0.80         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4


✅ Modellen och vektoriseraren är sparade!


In [5]:
# Testa modellen med nya meningar den aldrig sett förut
nya_meningar = [
    "Kan du lägga in ett läkarbesök för Pappa imorgon?", # Borde bli 1
    "Vilken dag töms papperskorgen?",                     # Borde bli 0
]

nya_meningar_vec = vectorizer.transform(nya_meningar)
prediktioner = model.predict(nya_meningar_vec)

# Ändrade 'i' till 'in' på raden nedan
for mening, pred in zip(nya_meningar, prediktioner):
    intent = "Kalenderhändelse" if pred == 1 else "Ogiltigt/Annat"
    print(f"Text: '{mening}' -> AI:ns slutsats: {intent}")

Text: 'Kan du lägga in ett läkarbesök för Pappa imorgon?' -> AI:ns slutsats: Kalenderhändelse
Text: 'Vilken dag töms papperskorgen?' -> AI:ns slutsats: Ogiltigt/Annat


In [6]:
import re
from datetime import datetime, timedelta

def extrahera_kalender_info(text):
    text = text.lower() # Gör all text till små bokstäver för enklare sökning
    
    # 1. Vilka är familjemedlemmarna?
    familj = ["mamma", "pappa", "robin", "colin", "mio", "mika"]
    hittad_person = "Okänd"
    for person in familj:
        if person in text:
            hittad_person = person.capitalize()
            break # Vi antar att det bara är en person per händelse just nu
            
    # 2. När ska det ske? (Enkel regelbaserad logik för MVP)
    datum = "Okänt"
    idag = datetime.now()
    if "imorgon" in text:
        datum = (idag + timedelta(days=1)).strftime("%Y-%m-%d")
    elif "idag" in text or "ikväll" in text:
        datum = idag.strftime("%Y-%m-%d")
    # Här kan du lägga till fler dagar senare (t.ex. "på fredag")
    
    # 3. Vad ska hända? (Allt som är kvar när vi plockat bort nyckelord)
    # Detta är en väldigt grov förenkling för att komma igång
    skrap_ord = ["lägg", "in", "till", "en", "ett", "för", "kan", "du", "påminn", "mig", "att"]
    rensad_text = text
    for ord in skrap_ord + familj + ["imorgon", "idag", "ikväll"]:
        rensad_text = re.sub(rf"\b{ord}\b", "", rensad_text).strip()
    
    # Ta bort extra mellanslag som kan ha uppstått
    hittad_aktivitet = " ".join(rensad_text.split()).capitalize()
    
    return {
        "person": hittad_person,
        "aktivitet": hittad_aktivitet if hittad_aktivitet else "Ny händelse",
        "datum": datum
    }

# Låt oss testa vår nya funktion!
test_mening = "Kan du lägga in ett läkarbesök för Pappa imorgon?"
resultat = extrahera_kalender_info(test_mening)

print(f"Ursprunglig text: '{test_mening}'")
print(f"Extraherat: Vem={resultat['person']}, Vad={resultat['aktivitet']}, När={resultat['datum']}")

Ursprunglig text: 'Kan du lägga in ett läkarbesök för Pappa imorgon?'
Extraherat: Vem=Pappa, Vad=Lägga läkarbesök ?, När=2026-04-21


In [ ]:
from openai import OpenAI

client = OpenAI(
  api_key="api_nyckel_borttagen_för_säkerhet_kan_applicera_json"
)

response = client.responses.create(
  model="gpt-5-nano",
  input="write a haiku about ai",
  store=True,
)

print(response.output_text);


Silent circuits dream
Drawing meaning from data
A mirror of ours


In [ ]:
import json
import joblib
from openai import OpenAI

# Ladda din lokala modell och vectorizer
laddad_modell = joblib.load("intent_model.pkl")
laddad_vectorizer = joblib.load("vectorizer.pkl")

# Lägg in din riktiga API-nyckel här
client = OpenAI(api_key="api_nyckel_borttagen_för_säkerhet_kan_applicera_json")

def tvatta_text(text):
    ersattningar = {
        "å": "a", "ä": "a", "ö": "o",
        "Å": "A", "Ä": "A", "Ö": "O"
    }
    for gammal, ny in ersattningar.items():
        text = text.replace(gammal, ny)
    return text

def ai_extrahera_kalender(text):
    saker_text = tvatta_text(text)

    system_instruktion = """
You are Eda, a family calendar assistant.
Your only job is to extract structured calendar data from the user's sentence.

Return ONLY a valid JSON object with these keys:
- person_id
- title
- event_time

Rules:
- person_id:
  1 = Mamma
  2 = Pappa
  3 = Robin
  4 = Colin
  5 = Mio
  6 = Mika
  0 = unknown person

- title:
  A short Swedish event title.

- event_time:
  Return format YYYY-MM-DD HH:MM when possible.
  If user says "imorgon", use 2026-04-21.
  If no time is given but date is understood, use 12:00.
  If nothing time-related is understood, return "unknown".

Examples:
{"person_id": 2, "title": "Lakarbesok", "event_time": "2026-04-21 14:00"}
{"person_id": 4, "title": "Fotboll", "event_time": "2026-04-21 12:00"}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_instruktion},
                {"role": "user", "content": saker_text}
            ],
            temperature=0
        )

        svar_strang = response.choices[0].message.content
        kalender_data = json.loads(svar_strang)
        return kalender_data

    except Exception as e:
        return {"error": f"API-fel: {e}"}

def edas_huvudlogik(anvandar_text):
    print(f"\nDu sa: '{anvandar_text}'")

    text_vec = laddad_vectorizer.transform([anvandar_text])
    ar_kalender = laddad_modell.predict(text_vec)[0]

    if ar_kalender == 0:
        return "Eda: Jag forstar inte, eller sa hanterar jag inte det amnet. Jag hanterar bara kalendern."

    print("Eda: Uppfattat! Tolkar detaljerna...")

    extraherad_data = ai_extrahera_kalender(anvandar_text)

    if "error" in extraherad_data:
        return f"Eda: Kunde inte tolka datan. Fel: {extraherad_data['error']}"

    sql_query = (
        f"INSERT INTO event (person_id, title, event_time) "
        f"VALUES ({extraherad_data.get('person_id')}, "
        f"'{extraherad_data.get('title')}', "
        f"'{extraherad_data.get('event_time')}');"
    )

    print(f"Klar att kora SQL: {sql_query}")
    return "Eda: Handelsen ar tillagd!"

# Test 1
svar1 = edas_huvudlogik("Kan du lagga in ett lakarbesok for Pappa imorgon klockan 14?")
print(svar1)

# Test 2
svar2 = edas_huvudlogik("Vad blir det for vader imorgon?")
print(svar2)


Du sa: 'Kan du lagga in ett lakarbesok for Pappa imorgon klockan 14?'
Eda: Uppfattat! Tolkar detaljerna...
Klar att kora SQL: INSERT INTO event (person_id, title, event_time) VALUES (2, 'Lakarbesok', '2026-04-21 14:00');
Eda: Handelsen ar tillagd!

Du sa: 'Vad blir det for vader imorgon?'
Eda: Jag forstar inte, eller sa hanterar jag inte det amnet. Jag hanterar bara kalendern.


In [9]:
# körd i cmd isetället !conda install -c conda-forge ffmpeg -y

In [11]:
import os
import subprocess

# Leta upp FFmpeg via Windows kommandotolk
try:
    sökväg = subprocess.check_output("where ffmpeg", shell=True).decode('utf-8').strip()
    print(f"Hittade FFmpeg här:\n{sökväg}")
except Exception as e:
    print("Kunde inte hitta FFmpeg via 'where' kommandot.")

Hittade FFmpeg här:
C:\Users\tommi\anaconda3\envs\tf210\Library\bin\ffmpeg.exe


In [14]:
!pip install soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 1.4 MB/s eta 0:00:01
   ------------------------------ --------- 0.8/1.0 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 1.6 MB/s  0:00:01


In [20]:
import whisper
import sounddevice as sd
import numpy as np
import time

print("Laddar Whisper-modellen...")
modell_whisper = whisper.load_model("base")
print("Modellen är redo!")

duration = 5  # sekunder
sample_rate = 16000  # 16 kHz

print(f"\n🎤 Spelar in i {duration} sekunder... PRATA TYDLIGT!")
audio_data = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
sd.wait()

# Platta till arrayen
audio_flat = audio_data.flatten()

# Ljud-diagnostik: Kontrollera att mikrofonen fungerar
max_volym = np.max(np.abs(audio_flat))
print(f"📊 Max ljudnivå uppmätt: {max_volym:.4f}")

if max_volym < 0.01:
    print("⚠️ Ljudnivån är extremt låg. Mikrofonen kanske är mutad eller vald till fel enhet i Windows!")
else:
    print("✅ Ljud fångat! Analyserar...")
    # Normalisera ljudet så det ligger mellan -1.0 och 1.0 (det Whisper föredrar)
    audio_flat = audio_flat / max_volym
    
    # Kör Whisper med parametern fp16=False för att undvika varningar på vissa grafikkort
    resultat = modell_whisper.transcribe(audio_flat, language="sv", fp16=False)
    print(f"\n🗣️ Eda hörde: '{resultat['text'].strip()}'")

Laddar Whisper-modellen...
Modellen är redo!

🎤 Spelar in i 5 sekunder... PRATA TYDLIGT!
📊 Max ljudnivå uppmätt: 0.5969
✅ Ljud fångat! Analyserar...

🗣️ Eda hörde: 'test'


In [ ]:
import whisper
import sounddevice as sd
import numpy as np
import time
import json
import joblib
from openai import OpenAI

# =====================================================================
# 1. SETUP & INITIALISERING (Ladda alla AI-modeller i minnet)
# =====================================================================
import json

print("Laddar hela Eda-systemet (Detta tar en stund)...")

# Ladda API-nyckeln säkert från config.json
try:
    with open("config.json", "r") as config_file:
        config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]
except FileNotFoundError:
    print("❌ FEL: Hittade inte config.json. Skapa filen och lägg in din OPENAI_API_KEY.")
    api_key = None # Eller hantera felet på ett sätt som passar dig

# OpenAI API-nyckel för Hjärnan
client = OpenAI(api_key=api_key)

# Ladda Whisper (Öronen)
modell_whisper = whisper.load_model("base")

# Ladda din egenbyggda maskininlärningsmodell (Filtret/Dörrvakten)
laddad_modell = joblib.load("intent_model.pkl")
laddad_vectorizer = joblib.load("vectorizer.pkl")

print("✅ Alla AI-modeller är redo att användas!\n")


# =====================================================================
# 2. HJÄRNAN (Text-sanering och OpenAI-tolkning)
# =====================================================================
def tvatta_text(text):
    ersattningar = {"å": "a", "ä": "a", "ö": "o", "Å": "A", "Ä": "A", "Ö": "O"}
    for gammal, ny in ersattningar.items():
        text = text.replace(gammal, ny)
    return text

def ai_extrahera_kalender(text):
    saker_text = tvatta_text(text)
    
    # Uppdaterad och striktare prompt för att förhindra gissningar
    system_instruktion = """
    You are Eda, a family calendar assistant.
    Return ONLY a valid JSON object with these keys: person_id, title, event_time.
    
    Rules for person_id: 
    1=Mamma, 2=Pappa, 3=Robin, 4=Colin, 5=Mio, 6=Mika. 
    If the exact name or a very close phonetic match is NOT found in the text, you MUST return 0. Do not guess randomly.
    
    Rules for title: Short Swedish title (fix å,ä,ö).
    
    Rules for event_time: Format 'YYYY-MM-DD HH:MM'. If user says 'imorgon' use 2026-04-21.
    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_instruktion},
                {"role": "user", "content": saker_text}
            ],
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"error": f"API-fel: {e}"}

# =====================================================================
# 3. HUVUDPROGRAMMET (Lyssna -> Filtrera -> Tänk -> Agera)
# =====================================================================
def starta_eda_assistenten():
    # Spela in ljud
    duration = 5
    sample_rate = 16000
    print(f"🎤 Spelar in i {duration} sekunder... (T.ex. 'Lägg till fotboll för Colin klockan 18')")
    
    audio_data = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
    sd.wait()
    print("✅ Inspelning klar! Bearbetar...")
    
    audio_flat = audio_data.flatten()
    if np.max(np.abs(audio_flat)) < 0.01:
        print("🤖 Eda: Jag hörde ingenting, mikrofonen verkar vara för tyst.")
        return
        
    audio_flat = audio_flat / np.max(np.abs(audio_flat))
    
    # 1. Öronen: Whisper transkriberar med fusk-lapp!
    fusk_lapp = "Mamma, Pappa, Robin, Colin, Mio, Mika."
    resultat = modell_whisper.transcribe(
        audio_flat, 
        language="sv", 
        fp16=False,
        initial_prompt=fusk_lapp
    )
    
    anvandar_text = resultat['text'].strip()
    
    if not anvandar_text:
        print("🤖 Eda: Jag uppfattade inga ord. Kan du upprepa?")
        return
        
    print(f"\n🗣️ Du sa: '{anvandar_text}'")
    
    # 2. Filtret: Är detta ens en kalenderhändelse?
    text_vec = laddad_vectorizer.transform([anvandar_text])
    ar_kalender = laddad_modell.predict(text_vec)[0]
    
    if ar_kalender == 0:
        print("🤖 Eda: Jag förstår inte, eller så hanterar jag inte det ämnet. Jag hanterar bara kalendern!")
        return
        
    print("🤖 Eda: Uppfattat! Tolkar detaljerna...")
    
    # 3. Hjärnan: Extrahera data
    extraherad_data = ai_extrahera_kalender(anvandar_text)
    
    if "error" in extraherad_data:
        print(f"🤖 Eda: Kunde inte tolka datan. Fel: {extraherad_data['error']}")
        return
        
    # Säkerhetskontroll: Kastade OpenAI ut person=0 för att det var otydligt?
    if extraherad_data.get('person_id') == 0:
        print("🤖 Eda: Jag kunde tyvärr inte uppfatta vem i familjen händelsen gällde. Försök igen!")
        return
        
    # 4. Agera: Generera SQL
    sql_query = (
        f"INSERT INTO event (person_id, title, event_time) "
        f"VALUES ({extraherad_data.get('person_id')}, "
        f"'{extraherad_data.get('title')}', "
        f"'{extraherad_data.get('event_time')}');"
    )
    
    print(f"✅ Klar att köra SQL: {sql_query}")
    print("🤖 Eda: Händelsen är tillagd!")

# Kör igång!
starta_eda_assistenten()

Laddar hela Eda-systemet (Detta tar en stund)...
✅ Alla AI-modeller är redo att användas!

🎤 Spelar in i 5 sekunder... (T.ex. 'Lägg till fotboll för Colin klockan 18')
✅ Inspelning klar! Bearbetar...

🗣️ Du sa: 'Lägg till fotboll för Colin, klok en av dem.'
🤖 Eda: Uppfattat! Tolkar detaljerna...
✅ Klar att köra SQL: INSERT INTO event (person_id, title, event_time) VALUES (4, 'Fotboll', '2026-04-21 13:00');
🤖 Eda: Händelsen är tillagd!


In [27]:
!pip install pygame

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/10.6 MB 5.6 MB/s eta 0:00:02
   ----------------- ---------------------- 4.7/10.6 MB 11.0 MB/s eta 0:00:01
   ---------------------- ----------------- 6.0/10.6 MB 12.7 MB/s eta 0:00:01
   ---------------------------------------  10.5/10.6 MB 12.8 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 12.1 MB/s  0:00:01


In [ ]:
import pygame
from openai import OpenAI
import time
import os

client = OpenAI(api_key="api_nyckel_borttagen_för_säkerhet_kan_applicera_json")

def eda_pratar(text_att_saga):
    print(f"🔊 Eda säger: '{text_att_saga}'")
    ljudfil = "eda_svar.mp3"
    
    try:
        # 1. Anropa OpenAI för TTS på det moderna sättet
        # Vi använder en kontext-hanterare ('with') för att hantera minnet säkert
        with client.audio.speech.with_streaming_response.create(
            model="tts-1",
            voice="nova",
            input=text_att_saga
        ) as response:
            # 2. Skriv direkt till fil istället för att använda det trasiga stream_to_file
            response.stream_to_file(ljudfil) # OBS! OpenAI har lagat detta i nyaste versionen genom att just tvinga dig använda blocket ovan
            
        # 3. Spela upp ljudet
        pygame.mixer.init()
        pygame.mixer.music.load(ljudfil)
        pygame.mixer.music.play()
        
        while pygame.mixer.music.get_busy():
            time.sleep(0.1)
            
        pygame.mixer.quit()
        
        # Valfritt (men snyggt!): Radera mp3-filen när den är uppspelad så vi inte fyller hårddisken
        if os.path.exists(ljudfil):
            os.remove(ljudfil)
            
    except Exception as e:
        print(f"Kunde inte spela upp ljudet. Fel: {e}")

# Testa igen! Nu bör varningen vara borta.
eda_pratar("Jag hanterar din familjs kalender smidigare än någonsin!")

🔊 Eda säger: 'Jag hanterar din familjs kalender smidigare än någonsin!'


In [ ]:
import whisper
import sounddevice as sd
import numpy as np
import time
import json
import joblib
from openai import OpenAI
import pygame
import os

# =====================================================================
# 1. SETUP (Ladda allt i minnet för omedelbar respons)
# =====================================================================
print("Laddar hela Eda-systemet...")

# OpenAI-klient (För Hjärnan och Munnen)
client = OpenAI(api_key="api_nyckel_borttagen_för_säkerhet_kan_applicera_json")

# Ladda Whisper (Öronen)
modell_whisper = whisper.load_model("base")

# Ladda din egenbyggda ML-modell (Filtret)
laddad_modell = joblib.load("intent_model.pkl")
laddad_vectorizer = joblib.load("vectorizer.pkl")

print("✅ Systemet är redo! Tryck på kör-knappen när du vill prata.\n")

# =====================================================================
# 2. HJÄLPFUNKTIONER (Hjärnan och Munnen)
# =====================================================================
def tvatta_text(text):
    ersattningar = {"å": "a", "ä": "a", "ö": "o", "Å": "A", "Ä": "A", "Ö": "O"}
    for gammal, ny in ersattningar.items():
        text = text.replace(gammal, ny)
    return text

def ai_extrahera_kalender(text):
    saker_text = tvatta_text(text)
    
    system_instruktion = """
    You are Eda, a family calendar assistant.
    Return ONLY a valid JSON object with these keys: person_id, title, event_time.
    
    Rules for person_id: 
    1=Mamma, 2=Pappa, 3=Robin, 4=Colin, 5=Mio, 6=Mika. 
    If the exact name or a very close phonetic match is NOT found in the text, you MUST return 0.
    
    Rules for title: Short Swedish title (fix å,ä,ö).
    
    Rules for event_time: Format 'YYYY-MM-DD HH:MM'. If user says 'imorgon' use 2026-04-21.
    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_instruktion},
                {"role": "user", "content": saker_text}
            ],
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"error": f"API-fel: {e}"}

def eda_pratar(text_att_saga):
    print(f"🔊 Eda säger: '{text_att_saga}'")
    ljudfil = "eda_svar.mp3"
    
    try:
        with client.audio.speech.with_streaming_response.create(
            model="tts-1",
            voice="nova",
            input=text_att_saga
        ) as response:
            response.stream_to_file(ljudfil)
            
        pygame.mixer.init()
        pygame.mixer.music.load(ljudfil)
        pygame.mixer.music.play()
        
        while pygame.mixer.music.get_busy():
            time.sleep(0.1)
            
        pygame.mixer.quit()
        if os.path.exists(ljudfil):
            os.remove(ljudfil)
            
    except Exception as e:
        print(f"Kunde inte spela upp ljudet. Fel: {e}")

# =====================================================================
# 3. HUVUDPROGRAMMET (Den stora assistenten)
# =====================================================================
def klicka_pa_mikrofonen():
    duration = 5
    sample_rate = 16000
    print(f"🎤 Spelar in ljud... PRATA NU! (Du har 5 sekunder på dig)")
    
    audio_data = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
    sd.wait()
    print("✅ Inspelning klar! Eda tänker...")
    
    audio_flat = audio_data.flatten()
    if np.max(np.abs(audio_flat)) < 0.01:
        eda_pratar("Jag hörde ingenting. Mikrofonen verkar vara för tyst.")
        return
        
    audio_flat = audio_flat / np.max(np.abs(audio_flat))
    
    # 1. Lyssna med Whisper (Öronen)
    fusk_lapp = "Mamma, Pappa, Robin, Colin, Mio, Mika."
    resultat = modell_whisper.transcribe(audio_flat, language="sv", fp16=False, initial_prompt=fusk_lapp)
    
    anvandar_text = resultat['text'].strip()
    if not anvandar_text:
        eda_pratar("Ursäkta, jag uppfattade inga ord. Kan du upprepa?")
        return
        
    print(f"\n🗣️ Du sa: '{anvandar_text}'")
    
    # 2. Din egen ML-modell bestämmer (Filtret)
    text_vec = laddad_vectorizer.transform([anvandar_text])
    ar_kalender = laddad_modell.predict(text_vec)[0]
    
    if ar_kalender == 0:
        eda_pratar("Jag hanterar bara kalendern, tyvärr! Fråga mig om att lägga till händelser istället.")
        return
        
    # 3. Extrahera data till kalendern (Hjärnan)
    extraherad_data = ai_extrahera_kalender(anvandar_text)
    
    if "error" in extraherad_data:
        eda_pratar(f"Kunde inte tolka datan. Fel: {extraherad_data['error']}")
        return
        
    person = extraherad_data.get('person_id')
    titel = extraherad_data.get('title')
    
    if person == 0:
        eda_pratar("Jag kunde tyvärr inte uppfatta vem i familjen händelsen gällde. Försök igen!")
        return
        
    # 4. Generera SQL-koden för dashboarden
    sql_query = (
        f"INSERT INTO event (person_id, title, event_time) "
        f"VALUES ({person}, '{titel}', '{extraherad_data.get('event_time')}');"
    )
    
    print(f"✅ Klar att köra SQL: {sql_query}")
    
    # 5. Bekräfta med den nya fantastiska rösten (Munnen)
    eda_pratar(f"Jag har nu lagt till {titel} i kalendern!")

# Nu trycker vi på knappen! Säg en kalenderhändelse, eller fråga om vädret.
klicka_pa_mikrofonen()

Laddar hela Eda-systemet...
✅ Systemet är redo! Tryck på kör-knappen när du vill prata.

🎤 Spelar in ljud... PRATA NU! (Du har 5 sekunder på dig)
✅ Inspelning klar! Eda tänker...

🗣️ Du sa: 'Lägg in fotboll för Robin i Morgan klockan sjuk.'
✅ Klar att köra SQL: INSERT INTO event (person_id, title, event_time) VALUES (3, 'Fotboll', '2026-04-21 07:00');
🔊 Eda säger: 'Jag har nu lagt till Fotboll i kalendern!'


Alternativ C: Genomföra ett valfritt projekt inom AI/IoT


I slutet av din kod ska du redogöra för hur din modell hade kunnat användas i 
verkligheten och vilka potentiella utmaningar och möjligheter (tex affärsmässiga, etiska 
och andra perspektiv du finner relevanta) som finns. Du kan skriva detta som 
kommentarer i koden.


Min reflektion kring ovanstående:

Användning i verkligheten och affärsmässiga möjligheter: Logistik och lager situationer där en mer väl utvecklad modell kan brukas av personal som exempelvis kan rapportera in lagersaldon eller avvikelser under arbetet, handsfree. Kanske direkt in i databas/BI-system som företaget använder.

Annat användningsområde skulle kunna vara inom vård och omsorg. Kanske som en kombination av bokning av resurser och diktering av loggar och liknande under arbete som pågår fysiskt med patienter.

Utmaningar, även om en modell är mer välgjort än min egen, skulle ändå kunna bli svårigheter kring stökiga arbetsmiljöer där bakgrundsbrus, personer som pratar i bakgrunden samtidigt eller kanske dialekter. Något som jag själv snabbt märkte var ett hinder då extremt tydligt tal är en viktig del för att få rätt output. Nu finns det bättre modeller, men det säger ändå något om problematik som kan uppstå där det absolut inte finns utrymme till sådant.

Gällande etiska/integritetsmässiga perspektiv så behöver jag nog inte utveckla allt för mycket. Det är röst/ljud som spelas in och med det kommer känslighet som under samtal. GDPR är också något som måste tänkas på. Datasäkerhet i sig gällande data till tredjepart som cloud tjänster. Samtycke bland anställda är en viktig punkt.